<a href="https://colab.research.google.com/github/Ishkei/Binaural-Doppler-Technique-Intense-Version-/blob/main/Binaural_Doppler_Technique_(Intense).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import wave

# --- Constants and Parameters ---
sample_rate = 44100  # Samples per second
duration = 3600  # Total duration in seconds (60 minutes)
n_channels = 2  # Stereo
sampwidth = 2  # 16-bit PCM (2 bytes per sample)
output_filename = 'binaural_entrainment_hour.wav'

# Frequency design parameters
base_freq = 108  # Starting frequency in Hz for both ears
freq_shift = 30.0  # Amount of frequency shift (±20 Hz)
midpoint = duration / 2.0  # 1800 seconds; at midpoint, left is at 56 Hz and right at 16 Hz

# Fade parameters (in seconds)
fade_duration = 10.0  # 10-second fade-in and fade-out

# Maximum amplitude (scaled to 90% of full scale to prevent clipping)
max_amplitude = 32767 * 0.9

# --- Frequency Functions ---
def left_frequency(t):
    """ Returns the instantaneous left-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency increases linearly from base_freq to (base_freq+freq_shift).
    For midpoint < t <= duration, the frequency decreases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq + (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq + freq_shift) - (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


def right_frequency(t):
    """ Returns the instantaneous right-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency decreases linearly from base_freq to (base_freq-freq_shift).
    For midpoint < t <= duration, the frequency increases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq - (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq - freq_shift) + (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


# --- Envelope Function ---
def envelope(t, total_duration):
    """ Returns an amplitude envelope for time array t with a linear fade-in during the
    first fade_duration seconds and a fade-out during the final fade_duration seconds.
    """
    env = np.ones_like(t)  # Fade-in: scale linearly from 0 to 1 over fade_duration seconds
    mask_in = t < fade_duration
    env[mask_in] = t[mask_in] / fade_duration
    # Fade-out: scale linearly from 1 to 0 over the last fade_duration seconds
    mask_out = t > (total_duration - fade_duration)
    env[mask_out] = (total_duration - t[mask_out]) / fade_duration
    return env


# --- Prepare WAV File for Writing ---
wav_file = wave.open(output_filename, 'w')
wav_file.setnchannels(n_channels)
wav_file.setsampwidth(sampwidth)
wav_file.setframerate(sample_rate)

# --- Chunk Processing Setup ---
chunk_duration = 1.0  # Process audio 1 second at a time
chunk_samples = int(chunk_duration * sample_rate)  # Number of samples per chunk
total_chunks = int(duration / chunk_duration)

# Initialize phases to maintain continuity between chunks
phase_left = 0.0
phase_right = 0.0

# --- Main Processing Loop ---
for chunk_index in range(total_chunks):
    # Global time for this chunk (from t_start to t_start + chunk_duration)
    t_start = chunk_index * chunk_duration
    # Create a time array for the current chunk
    t_chunk = np.linspace(t_start, t_start + chunk_duration, chunk_samples, endpoint=False)

    # Get instantaneous frequencies for left and right channels
    f_left = left_frequency(t_chunk)
    f_right = right_frequency(t_chunk)

    # Calculate phase increments for each sample: d_phase = 2*pi*f/sample_rate
    dphase_left = 2 * np.pi * f_left / sample_rate
    dphase_right = 2 * np.pi * f_right / sample_rate

    # Accumulate phase increments to ensure continuous sine waves between chunks
    phase_array_left = phase_left + np.cumsum(dphase_left)
    phase_array_right = phase_right + np.cumsum(dphase_right)

    # Update the starting phase for the next chunk
    phase_left = phase_array_left[-1]
    phase_right = phase_array_right[-1]

    # Generate the sine waves for this chunk
    chunk_left = np.sin(phase_array_left)
    chunk_right = np.sin(phase_array_right)

    # Apply the fade envelope for smooth fade-in/out
    env = envelope(t_chunk, duration)
    chunk_left *= env
    chunk_right *= env

    # Scale the samples and convert to 16-bit integers
    chunk_left = (chunk_left * max_amplitude).astype(np.int16)
    chunk_right = (chunk_right * max_amplitude).astype(np.int16)

    # Interleave the two channels (stereo)
    interleaved = np.empty(chunk_samples * 2, dtype=np.int16)
    interleaved[0::2] = chunk_left
    interleaved[1::2] = chunk_right

    # Write the current chunk's frames to the WAV file
    wav_file.writeframes(interleaved.tobytes())

wav_file.close()
print("WAV file generated:", output_filename)
import numpy as np
import wave

# --- Constants and Parameters ---
sample_rate = 44100  # Samples per second
duration = 3600  # Total duration in seconds (60 minutes)
n_channels = 2  # Stereo
sampwidth = 2  # 16-bit PCM (2 bytes per sample)
output_filename = 'binaural_entrainment_hour.wav'

# Frequency design parameters
base_freq = 108  # Starting frequency in Hz for both ears
freq_shift = 30.0  # Amount of frequency shift (±20 Hz)
midpoint = duration / 2.0  # 1800 seconds; at midpoint, left is at 56 Hz and right at 16 Hz

# Fade parameters (in seconds)
fade_duration = 10.0  # 10-second fade-in and fade-out

# Maximum amplitude (scaled to 90% of full scale to prevent clipping)
max_amplitude = 32767 * 0.9

# --- Frequency Functions ---
def left_frequency(t):
    """ Returns the instantaneous left-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency increases linearly from base_freq to (base_freq+freq_shift).
    For midpoint < t <= duration, the frequency decreases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq + (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq + freq_shift) - (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


def right_frequency(t):
    """ Returns the instantaneous right-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency decreases linearly from base_freq to (base_freq-freq_shift).
    For midpoint < t <= duration, the frequency increases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq - (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq - freq_shift) + (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


# --- Envelope Function ---
def envelope(t, total_duration):
    """ Returns an amplitude envelope for time array t with a linear fade-in during the
    first fade_duration seconds and a fade-out during the final fade_duration seconds.
    """
    env = np.ones_like(t)  # Fade-in: scale linearly from 0 to 1 over fade_duration seconds
    mask_in = t < fade_duration
    env[mask_in] = t[mask_in] / fade_duration
    # Fade-out: scale linearly from 1 to 0 over the last fade_duration seconds
    mask_out = t > (total_duration - fade_duration)
    env[mask_out] = (total_duration - t[mask_out]) / fade_duration
    return env


# --- Prepare WAV File for Writing ---
wav_file = wave.open(output_filename, 'w')
wav_file.setnchannels(n_channels)
wav_file.setsampwidth(sampwidth)
wav_file.setframerate(sample_rate)

# --- Chunk Processing Setup ---
chunk_duration = 1.0  # Process audio 1 second at a time
chunk_samples = int(chunk_duration * sample_rate)  # Number of samples per chunk
total_chunks = int(duration / chunk_duration)

# Initialize phases to maintain continuity between chunks
phase_left = 0.0
phase_right = 0.0

# --- Main Processing Loop ---
for chunk_index in range(total_chunks):
    # Global time for this chunk (from t_start to t_start + chunk_duration)
    t_start = chunk_index * chunk_duration
    # Create a time array for the current chunk
    t_chunk = np.linspace(t_start, t_start + chunk_duration, chunk_samples, endpoint=False)

    # Get instantaneous frequencies for left and right channels
    f_left = left_frequency(t_chunk)
    f_right = right_frequency(t_chunk)

    # Calculate phase increments for each sample: d_phase = 2*pi*f/sample_rate
    dphase_left = 2 * np.pi * f_left / sample_rate
    dphase_right = 2 * np.pi * f_right / sample_rate

    # Accumulate phase increments to ensure continuous sine waves between chunks
    phase_array_left = phase_left + np.cumsum(dphase_left)
    phase_array_right = phase_right + np.cumsum(dphase_right)

    # Update the starting phase for the next chunk
    phase_left = phase_array_left[-1]
    phase_right = phase_array_right[-1]

    # Generate the sine waves for this chunk
    chunk_left = np.sin(phase_array_left)
    chunk_right = np.sin(phase_array_right)

    # Apply the fade envelope for smooth fade-in/out
    env = envelope(t_chunk, duration)
    chunk_left *= env
    chunk_right *= env

    # Scale the samples and convert to 16-bit integers
    chunk_left = (chunk_left * max_amplitude).astype(np.int16)
    chunk_right = (chunk_right * max_amplitude).astype(np.int16)

    # Interleave the two channels (stereo)
    interleaved = np.empty(chunk_samples * 2, dtype=np.int16)
    interleaved[0::2] = chunk_left
    interleaved[1::2] = chunk_right

    # Write the current chunk's frames to the WAV file
    wav_file.writeframes(interleaved.tobytes())

import numpy as np
import wave

# --- Constants and Parameters ---
sample_rate = 44100  # Samples per second
duration = 3600  # Total duration in seconds (60 minutes)
n_channels = 2  # Stereo
sampwidth = 2  # 16-bit PCM (2 bytes per sample)
output_filename = 'binaural_entrainment_hour.wav'

# Frequency design parameters
base_freq = 108  # Starting frequency in Hz for both ears
freq_shift = 30.0  # Amount of frequency shift (±20 Hz)
midpoint = duration / 2.0  # 1800 seconds; at midpoint, left is at 56 Hz and right at 16 Hz

# Fade parameters (in seconds)
fade_duration = 10.0  # 10-second fade-in and fade-out

# Maximum amplitude (scaled to 90% of full scale to prevent clipping)
max_amplitude = 32767 * 0.9

# --- Frequency Functions ---
def left_frequency(t):
    """ Returns the instantaneous left-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency increases linearly from base_freq to (base_freq+freq_shift).
    For midpoint < t <= duration, the frequency decreases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq + (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq + freq_shift) - (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


def right_frequency(t):
    """ Returns the instantaneous right-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency decreases linearly from base_freq to (base_freq-freq_shift).
    For midpoint < t <= duration, the frequency increases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq - (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq - freq_shift) + (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


# --- Envelope Function ---
def envelope(t, total_duration):
    """ Returns an amplitude envelope for time array t with a linear fade-in during the
    first fade_duration seconds and a fade-out during the final fade_duration seconds.
    """
    env = np.ones_like(t)  # Fade-in: scale linearly from 0 to 1 over fade_duration seconds
    mask_in = t < fade_duration
    env[mask_in] = t[mask_in] / fade_duration
    # Fade-out: scale linearly from 1 to 0 over the last fade_duration seconds
    mask_out = t > (total_duration - fade_duration)
    env[mask_out] = (total_duration - t[mask_out]) / fade_duration
    return env


# --- Prepare WAV File for Writing ---
wav_file = wave.open(output_filename, 'w')
wav_file.setnchannels(n_channels)
wav_file.setsampwidth(sampwidth)
wav_file.setframerate(sample_rate)

# --- Chunk Processing Setup ---
chunk_duration = 1.0  # Process audio 1 second at a time
chunk_samples = int(chunk_duration * sample_rate)  # Number of samples per chunk
total_chunks = int(duration / chunk_duration)

# Initialize phases to maintain continuity between chunks
phase_left = 0.0
phase_right = 0.0

# --- Main Processing Loop ---
for chunk_index in range(total_chunks):
    # Global time for this chunk (from t_start to t_start + chunk_duration)
    t_start = chunk_index * chunk_duration
    # Create a time array for the current chunk
    t_chunk = np.linspace(t_start, t_start + chunk_duration, chunk_samples, endpoint=False)

    # Get instantaneous frequencies for left and right channels
    f_left = left_frequency(t_chunk)
    f_right = right_frequency(t_chunk)

    # Calculate phase increments for each sample: d_phase = 2*pi*f/sample_rate
    dphase_left = 2 * np.pi * f_left / sample_rate
    dphase_right = 2 * np.pi * f_right / sample_rate

    # Accumulate phase increments to ensure continuous sine waves between chunks
    phase_array_left = phase_left + np.cumsum(dphase_left)
    phase_array_right = phase_right + np.cumsum(dphase_right)

    # Update the starting phase for the next chunk
    phase_left = phase_array_left[-1]
    phase_right = phase_array_right[-1]

    # Generate the sine waves for this chunk
    chunk_left = np.sin(phase_array_left)
    chunk_right = np.sin(phase_array_right)

    # Apply the fade envelope for smooth fade-in/out
    env = envelope(t_chunk, duration)
    chunk_left *= env
    chunk_right *= env

    # Scale the samples and convert to 16-bit integers
    chunk_left = (chunk_left * max_amplitude).astype(np.int16)
    chunk_right = (chunk_right * max_amplitude).astype(np.int16)

    # Interleave the two channels (stereo)
    interleaved = np.empty(chunk_samples * 2, dtype=np.int16)
    interleaved[0::2] = chunk_left
    interleaved[1::2] = chunk_right

    # Write the current chunk's frames to the WAV file
    wav_file.writeframes(interleaved.tobytes())

import numpy as np
import wave

# --- Constants and Parameters ---
sample_rate = 44100  # Samples per second
duration = 3600  # Total duration in seconds (60 minutes)
n_channels = 2  # Stereo
sampwidth = 2  # 16-bit PCM (2 bytes per sample)
output_filename = 'binaural_entrainment_hour.wav'

# Frequency design parameters
base_freq = 108  # Starting frequency in Hz for both ears
freq_shift = 30.0  # Amount of frequency shift (±20 Hz)
midpoint = duration / 2.0  # 1800 seconds; at midpoint, left is at 56 Hz and right at 16 Hz

# Fade parameters (in seconds)
fade_duration = 10.0  # 10-second fade-in and fade-out

# Maximum amplitude (scaled to 90% of full scale to prevent clipping)
max_amplitude = 32767 * 0.9

# --- Frequency Functions ---
def left_frequency(t):
    """ Returns the instantaneous left-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency increases linearly from base_freq to (base_freq+freq_shift).
    For midpoint < t <= duration, the frequency decreases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq + (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq + freq_shift) - (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


def right_frequency(t):
    """ Returns the instantaneous right-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency decreases linearly from base_freq to (base_freq-freq_shift).
    For midpoint < t <= duration, the frequency increases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq - (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq - freq_shift) + (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


# --- Envelope Function ---
def envelope(t, total_duration):
    """ Returns an amplitude envelope for time array t with a linear fade-in during the
    first fade_duration seconds and a fade-out during the final fade_duration seconds.
    """
    env = np.ones_like(t)  # Fade-in: scale linearly from 0 to 1 over fade_duration seconds
    mask_in = t < fade_duration
    env[mask_in] = t[mask_in] / fade_duration
    # Fade-out: scale linearly from 1 to 0 over the last fade_duration seconds
    mask_out = t > (total_duration - fade_duration)
    env[mask_out] = (total_duration - t[mask_out]) / fade_duration
    return env


# --- Prepare WAV File for Writing ---
wav_file = wave.open(output_filename, 'w')
wav_file.setnchannels(n_channels)
wav_file.setsampwidth(sampwidth)
wav_file.setframerate(sample_rate)

# --- Chunk Processing Setup ---
chunk_duration = 1.0  # Process audio 1 second at a time
chunk_samples = int(chunk_duration * sample_rate)  # Number of samples per chunk
total_chunks = int(duration / chunk_duration)

# Initialize phases to maintain continuity between chunks
phase_left = 0.0
phase_right = 0.0

# --- Main Processing Loop ---
for chunk_index in range(total_chunks):
    # Global time for this chunk (from t_start to t_start + chunk_duration)
    t_start = chunk_index * chunk_duration
    # Create a time array for the current chunk
    t_chunk = np.linspace(t_start, t_start + chunk_duration, chunk_samples, endpoint=False)

    # Get instantaneous frequencies for left and right channels
    f_left = left_frequency(t_chunk)
    f_right = right_frequency(t_chunk)

    # Calculate phase increments for each sample: d_phase = 2*pi*f/sample_rate
    dphase_left = 2 * np.pi * f_left / sample_rate
    dphase_right = 2 * np.pi * f_right / sample_rate

    # Accumulate phase increments to ensure continuous sine waves between chunks
    phase_array_left = phase_left + np.cumsum(dphase_left)
    phase_array_right = phase_right + np.cumsum(dphase_right)

    # Update the starting phase for the next chunk
    phase_left = phase_array_left[-1]
    phase_right = phase_array_right[-1]

    # Generate the sine waves for this chunk
    chunk_left = np.sin(phase_array_left)
    chunk_right = np.sin(phase_array_right)

    # Apply the fade envelope for smooth fade-in/out
    env = envelope(t_chunk, duration)
    chunk_left *= env
    chunk_right *= env

    # Scale the samples and convert to 16-bit integers
    chunk_left = (chunk_left * max_amplitude).astype(np.int16)
    chunk_right = (chunk_right * max_amplitude).astype(np.int16)

    # Interleave the two channels (stereo)
    interleaved = np.empty(chunk_samples * 2, dtype=np.int16)
    interleaved[0::2] = chunk_left
    interleaved[1::2] = chunk_right

    # Write the current chunk's frames to the WAV file
    wav_file.writeframes(interleaved.tobytes())

import numpy as np
import wave

# --- Constants and Parameters ---
sample_rate = 44100  # Samples per second
duration = 3600  # Total duration in seconds (60 minutes)
n_channels = 2  # Stereo
sampwidth = 2  # 16-bit PCM (2 bytes per sample)
output_filename = 'binaural_entrainment_hour.wav'

# Frequency design parameters
base_freq = 108  # Starting frequency in Hz for both ears
freq_shift = 30.0  # Amount of frequency shift (±20 Hz)
midpoint = duration / 2.0  # 1800 seconds; at midpoint, left is at 56 Hz and right at 16 Hz

# Fade parameters (in seconds)
fade_duration = 10.0  # 10-second fade-in and fade-out

# Maximum amplitude (scaled to 90% of full scale to prevent clipping)
max_amplitude = 32767 * 0.9

# --- Frequency Functions ---
def left_frequency(t):
    """ Returns the instantaneous left-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency increases linearly from base_freq to (base_freq+freq_shift).
    For midpoint < t <= duration, the frequency decreases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq + (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq + freq_shift) - (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


def right_frequency(t):
    """ Returns the instantaneous right-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency decreases linearly from base_freq to (base_freq-freq_shift).
    For midpoint < t <= duration, the frequency increases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq - (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq - freq_shift) + (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


# --- Envelope Function ---
def envelope(t, total_duration):
    """ Returns an amplitude envelope for time array t with a linear fade-in during the
    first fade_duration seconds and a fade-out during the final fade_duration seconds.
    """
    env = np.ones_like(t)  # Fade-in: scale linearly from 0 to 1 over fade_duration seconds
    mask_in = t < fade_duration
    env[mask_in] = t[mask_in] / fade_duration
    # Fade-out: scale linearly from 1 to 0 over the last fade_duration seconds
    mask_out = t > (total_duration - fade_duration)
    env[mask_out] = (total_duration - t[mask_out]) / fade_duration
    return env


# --- Prepare WAV File for Writing ---
wav_file = wave.open(output_filename, 'w')
wav_file.setnchannels(n_channels)
wav_file.setsampwidth(sampwidth)
wav_file.setframerate(sample_rate)

# --- Chunk Processing Setup ---
chunk_duration = 1.0  # Process audio 1 second at a time
chunk_samples = int(chunk_duration * sample_rate)  # Number of samples per chunk
total_chunks = int(duration / chunk_duration)

# Initialize phases to maintain continuity between chunks
phase_left = 0.0
phase_right = 0.0

# --- Main Processing Loop ---
for chunk_index in range(total_chunks):
    # Global time for this chunk (from t_start to t_start + chunk_duration)
    t_start = chunk_index * chunk_duration
    # Create a time array for the current chunk
    t_chunk = np.linspace(t_start, t_start + chunk_duration, chunk_samples, endpoint=False)

    # Get instantaneous frequencies for left and right channels
    f_left = left_frequency(t_chunk)
    f_right = right_frequency(t_chunk)

    # Calculate phase increments for each sample: d_phase = 2*pi*f/sample_rate
    dphase_left = 2 * np.pi * f_left / sample_rate
    dphase_right = 2 * np.pi * f_right / sample_rate

    # Accumulate phase increments to ensure continuous sine waves between chunks
    phase_array_left = phase_left + np.cumsum(dphase_left)
    phase_array_right = phase_right + np.cumsum(dphase_right)

    # Update the starting phase for the next chunk
    phase_left = phase_array_left[-1]
    phase_right = phase_array_right[-1]

    # Generate the sine waves for this chunk
    chunk_left = np.sin(phase_array_left)
    chunk_right = np.sin(phase_array_right)

    # Apply the fade envelope for smooth fade-in/out
    env = envelope(t_chunk, duration)
    chunk_left *= env
    chunk_right *= env

    # Scale the samples and convert to 16-bit integers
    chunk_left = (chunk_left * max_amplitude).astype(np.int16)
    chunk_right = (chunk_right * max_amplitude).astype(np.int16)

    # Interleave the two channels (stereo)
    interleaved = np.empty(chunk_samples * 2, dtype=np.int16)
    interleaved[0::2] = chunk_left
    interleaved[1::2] = chunk_right

    # Write the current chunk's frames to the WAV file
    wav_file.writeframes(interleaved.tobytes())

import numpy as np
import wave

# --- Constants and Parameters ---
sample_rate = 44100  # Samples per second
duration = 3600  # Total duration in seconds (60 minutes)
n_channels = 2  # Stereo
sampwidth = 2  # 16-bit PCM (2 bytes per sample)
output_filename = 'binaural_entrainment_hour.wav'

# Frequency design parameters
base_freq = 108  # Starting frequency in Hz for both ears
freq_shift = 30.0  # Amount of frequency shift (±20 Hz)
midpoint = duration / 2.0  # 1800 seconds; at midpoint, left is at 56 Hz and right at 16 Hz

# Fade parameters (in seconds)
fade_duration = 10.0  # 10-second fade-in and fade-out

# Maximum amplitude (scaled to 90% of full scale to prevent clipping)
max_amplitude = 32767 * 0.9

# --- Frequency Functions ---
def left_frequency(t):
    """ Returns the instantaneous left-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency increases linearly from base_freq to (base_freq+freq_shift).
    For midpoint < t <= duration, the frequency decreases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq + (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq + freq_shift) - (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


def right_frequency(t):
    """ Returns the instantaneous right-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency decreases linearly from base_freq to (base_freq-freq_shift).
    For midpoint < t <= duration, the frequency increases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq - (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq - freq_shift) + (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


# --- Envelope Function ---
def envelope(t, total_duration):
    """ Returns an amplitude envelope for time array t with a linear fade-in during the
    first fade_duration seconds and a fade-out during the final fade_duration seconds.
    """
    env = np.ones_like(t)  # Fade-in: scale linearly from 0 to 1 over fade_duration seconds
    mask_in = t < fade_duration
    env[mask_in] = t[mask_in] / fade_duration
    # Fade-out: scale linearly from 1 to 0 over the last fade_duration seconds
    mask_out = t > (total_duration - fade_duration)
    env[mask_out] = (total_duration - t[mask_out]) / fade_duration
    return env


# --- Prepare WAV File for Writing ---
wav_file = wave.open(output_filename, 'w')
wav_file.setnchannels(n_channels)
wav_file.setsampwidth(sampwidth)
wav_file.setframerate(sample_rate)

# --- Chunk Processing Setup ---
chunk_duration = 1.0  # Process audio 1 second at a time
chunk_samples = int(chunk_duration * sample_rate)  # Number of samples per chunk
total_chunks = int(duration / chunk_duration)

# Initialize phases to maintain continuity between chunks
phase_left = 0.0
phase_right = 0.0

# --- Main Processing Loop ---
for chunk_index in range(total_chunks):
    # Global time for this chunk (from t_start to t_start + chunk_duration)
    t_start = chunk_index * chunk_duration
    # Create a time array for the current chunk
    t_chunk = np.linspace(t_start, t_start + chunk_duration, chunk_samples, endpoint=False)

    # Get instantaneous frequencies for left and right channels
    f_left = left_frequency(t_chunk)
    f_right = right_frequency(t_chunk)

    # Calculate phase increments for each sample: d_phase = 2*pi*f/sample_rate
    dphase_left = 2 * np.pi * f_left / sample_rate
    dphase_right = 2 * np.pi * f_right / sample_rate

    # Accumulate phase increments to ensure continuous sine waves between chunks
    phase_array_left = phase_left + np.cumsum(dphase_left)
    phase_array_right = phase_right + np.cumsum(dphase_right)

    # Update the starting phase for the next chunk
    phase_left = phase_array_left[-1]
    phase_right = phase_array_right[-1]

    # Generate the sine waves for this chunk
    chunk_left = np.sin(phase_array_left)
    chunk_right = np.sin(phase_array_right)

    # Apply the fade envelope for smooth fade-in/out
    env = envelope(t_chunk, duration)
    chunk_left *= env
    chunk_right *= env

    # Scale the samples and convert to 16-bit integers
    chunk_left = (chunk_left * max_amplitude).astype(np.int16)
    chunk_right = (chunk_right * max_amplitude).astype(np.int16)

    # Interleave the two channels (stereo)
    interleaved = np.empty(chunk_samples * 2, dtype=np.int16)
    interleaved[0::2] = chunk_left
    interleaved[1::2] = chunk_right

    # Write the current chunk's frames to the WAV file
    wav_file.writeframes(interleaved.tobytes())

import numpy as np
import wave

# --- Constants and Parameters ---
sample_rate = 44100  # Samples per second
duration = 3600  # Total duration in seconds (60 minutes)
n_channels = 2  # Stereo
sampwidth = 2  # 16-bit PCM (2 bytes per sample)
output_filename = 'binaural_entrainment_hour.wav'

# Frequency design parameters
base_freq = 108  # Starting frequency in Hz for both ears
freq_shift = 30.0  # Amount of frequency shift (±20 Hz)
midpoint = duration / 2.0  # 1800 seconds; at midpoint, left is at 56 Hz and right at 16 Hz

# Fade parameters (in seconds)
fade_duration = 10.0  # 10-second fade-in and fade-out

# Maximum amplitude (scaled to 90% of full scale to prevent clipping)
max_amplitude = 32767 * 0.9

# --- Frequency Functions ---
def left_frequency(t):
    """ Returns the instantaneous left-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency increases linearly from base_freq to (base_freq+freq_shift).
    For midpoint < t <= duration, the frequency decreases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq + (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq + freq_shift) - (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


def right_frequency(t):
    """ Returns the instantaneous right-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency decreases linearly from base_freq to (base_freq-freq_shift).
    For midpoint < t <= duration, the frequency increases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq - (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq - freq_shift) + (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


# --- Envelope Function ---
def envelope(t, total_duration):
    """ Returns an amplitude envelope for time array t with a linear fade-in during the
    first fade_duration seconds and a fade-out during the final fade_duration seconds.
    """
    env = np.ones_like(t)  # Fade-in: scale linearly from 0 to 1 over fade_duration seconds
    mask_in = t < fade_duration
    env[mask_in] = t[mask_in] / fade_duration
    # Fade-out: scale linearly from 1 to 0 over the last fade_duration seconds
    mask_out = t > (total_duration - fade_duration)
    env[mask_out] = (total_duration - t[mask_out]) / fade_duration
    return env


# --- Prepare WAV File for Writing ---
wav_file = wave.open(output_filename, 'w')
wav_file.setnchannels(n_channels)
wav_file.setsampwidth(sampwidth)
wav_file.setframerate(sample_rate)

# --- Chunk Processing Setup ---
chunk_duration = 1.0  # Process audio 1 second at a time
chunk_samples = int(chunk_duration * sample_rate)  # Number of samples per chunk
total_chunks = int(duration / chunk_duration)

# Initialize phases to maintain continuity between chunks
phase_left = 0.0
phase_right = 0.0

# --- Main Processing Loop ---
for chunk_index in range(total_chunks):
    # Global time for this chunk (from t_start to t_start + chunk_duration)
    t_start = chunk_index * chunk_duration
    # Create a time array for the current chunk
    t_chunk = np.linspace(t_start, t_start + chunk_duration, chunk_samples, endpoint=False)

    # Get instantaneous frequencies for left and right channels
    f_left = left_frequency(t_chunk)
    f_right = right_frequency(t_chunk)

    # Calculate phase increments for each sample: d_phase = 2*pi*f/sample_rate
    dphase_left = 2 * np.pi * f_left / sample_rate
    dphase_right = 2 * np.pi * f_right / sample_rate

    # Accumulate phase increments to ensure continuous sine waves between chunks
    phase_array_left = phase_left + np.cumsum(dphase_left)
    phase_array_right = phase_right + np.cumsum(dphase_right)

    # Update the starting phase for the next chunk
    phase_left = phase_array_left[-1]
    phase_right = phase_array_right[-1]

    # Generate the sine waves for this chunk
    chunk_left = np.sin(phase_array_left)
    chunk_right = np.sin(phase_array_right)

    # Apply the fade envelope for smooth fade-in/out
    env = envelope(t_chunk, duration)
    chunk_left *= env
    chunk_right *= env

    # Scale the samples and convert to 16-bit integers
    chunk_left = (chunk_left * max_amplitude).astype(np.int16)
    chunk_right = (chunk_right * max_amplitude).astype(np.int16)

    # Interleave the two channels (stereo)
    interleaved = np.empty(chunk_samples * 2, dtype=np.int16)
    interleaved[0::2] = chunk_left
    interleaved[1::2] = chunk_right

    # Write the current chunk's frames to the WAV file
    wav_file.writeframes(interleaved.tobytes())

import numpy as np
import wave

# --- Constants and Parameters ---
sample_rate = 44100  # Samples per second
duration = 3600  # Total duration in seconds (60 minutes)
n_channels = 2  # Stereo
sampwidth = 2  # 16-bit PCM (2 bytes per sample)
output_filename = 'binaural_entrainment_hour.wav'

# Frequency design parameters
base_freq = 108  # Starting frequency in Hz for both ears
freq_shift = 30.0  # Amount of frequency shift (±20 Hz)
midpoint = duration / 2.0  # 1800 seconds; at midpoint, left is at 56 Hz and right at 16 Hz

# Fade parameters (in seconds)
fade_duration = 10.0  # 10-second fade-in and fade-out

# Maximum amplitude (scaled to 90% of full scale to prevent clipping)
max_amplitude = 32767 * 0.9

# --- Frequency Functions ---
def left_frequency(t):
    """ Returns the instantaneous left-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency increases linearly from base_freq to (base_freq+freq_shift).
    For midpoint < t <= duration, the frequency decreases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq + (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq + freq_shift) - (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


def right_frequency(t):
    """ Returns the instantaneous right-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency decreases linearly from base_freq to (base_freq-freq_shift).
    For midpoint < t <= duration, the frequency increases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq - (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq - freq_shift) + (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


# --- Envelope Function ---
def envelope(t, total_duration):
    """ Returns an amplitude envelope for time array t with a linear fade-in during the
    first fade_duration seconds and a fade-out during the final fade_duration seconds.
    """
    env = np.ones_like(t)  # Fade-in: scale linearly from 0 to 1 over fade_duration seconds
    mask_in = t < fade_duration
    env[mask_in] = t[mask_in] / fade_duration
    # Fade-out: scale linearly from 1 to 0 over the last fade_duration seconds
    mask_out = t > (total_duration - fade_duration)
    env[mask_out] = (total_duration - t[mask_out]) / fade_duration
    return env


# --- Prepare WAV File for Writing ---
wav_file = wave.open(output_filename, 'w')
wav_file.setnchannels(n_channels)
wav_file.setsampwidth(sampwidth)
wav_file.setframerate(sample_rate)

# --- Chunk Processing Setup ---
chunk_duration = 1.0  # Process audio 1 second at a time
chunk_samples = int(chunk_duration * sample_rate)  # Number of samples per chunk
total_chunks = int(duration / chunk_duration)

# Initialize phases to maintain continuity between chunks
phase_left = 0.0
phase_right = 0.0

# --- Main Processing Loop ---
for chunk_index in range(total_chunks):
    # Global time for this chunk (from t_start to t_start + chunk_duration)
    t_start = chunk_index * chunk_duration
    # Create a time array for the current chunk
    t_chunk = np.linspace(t_start, t_start + chunk_duration, chunk_samples, endpoint=False)

    # Get instantaneous frequencies for left and right channels
    f_left = left_frequency(t_chunk)
    f_right = right_frequency(t_chunk)

    # Calculate phase increments for each sample: d_phase = 2*pi*f/sample_rate
    dphase_left = 2 * np.pi * f_left / sample_rate
    dphase_right = 2 * np.pi * f_right / sample_rate

    # Accumulate phase increments to ensure continuous sine waves between chunks
    phase_array_left = phase_left + np.cumsum(dphase_left)
    phase_array_right = phase_right + np.cumsum(dphase_right)

    # Update the starting phase for the next chunk
    phase_left = phase_array_left[-1]
    phase_right = phase_array_right[-1]

    # Generate the sine waves for this chunk
    chunk_left = np.sin(phase_array_left)
    chunk_right = np.sin(phase_array_right)

    # Apply the fade envelope for smooth fade-in/out
    env = envelope(t_chunk, duration)
    chunk_left *= env
    chunk_right *= env

    # Scale the samples and convert to 16-bit integers
    chunk_left = (chunk_left * max_amplitude).astype(np.int16)
    chunk_right = (chunk_right * max_amplitude).astype(np.int16)

    # Interleave the two channels (stereo)
    interleaved = np.empty(chunk_samples * 2, dtype=np.int16)
    interleaved[0::2] = chunk_left
    interleaved[1::2] = chunk_right

    # Write the current chunk's frames to the WAV file
    wav_file.writeframes(interleaved.tobytes())

wav_file.close()
print("WAV file generated:", output_filename)
import numpy as np
import wave

# --- Constants and Parameters ---
sample_rate = 44100  # Samples per second
duration = 3600  # Total duration in seconds (60 minutes)
n_channels = 2  # Stereo
sampwidth = 2  # 16-bit PCM (2 bytes per sample)
output_filename = 'binaural_entrainment_hour.wav'

# Frequency design parameters
base_freq = 108  # Starting frequency in Hz for both ears
freq_shift = 30.0  # Amount of frequency shift (±20 Hz)
midpoint = duration / 2.0  # 1800 seconds; at midpoint, left is at 56 Hz and right at 16 Hz

# Fade parameters (in seconds)
fade_duration = 10.0  # 10-second fade-in and fade-out

# Maximum amplitude (scaled to 90% of full scale to prevent clipping)
max_amplitude = 32767 * 0.9

# --- Frequency Functions ---
def left_frequency(t):
    """ Returns the instantaneous left-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency increases linearly from base_freq to (base_freq+freq_shift).
    For midpoint < t <= duration, the frequency decreases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq + (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq + freq_shift) - (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


def right_frequency(t):
    """ Returns the instantaneous right-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency decreases linearly from base_freq to (base_freq-freq_shift).
    For midpoint < t <= duration, the frequency increases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq - (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq - freq_shift) + (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


# --- Envelope Function ---
def envelope(t, total_duration):
    """ Returns an amplitude envelope for time array t with a linear fade-in during the
    first fade_duration seconds and a fade-out during the final fade_duration seconds.
    """
    env = np.ones_like(t)  # Fade-in: scale linearly from 0 to 1 over fade_duration seconds
    mask_in = t < fade_duration
    env[mask_in] = t[mask_in] / fade_duration
    # Fade-out: scale linearly from 1 to 0 over the last fade_duration seconds
    mask_out = t > (total_duration - fade_duration)
    env[mask_out] = (total_duration - t[mask_out]) / fade_duration
    return env


# --- Prepare WAV File for Writing ---
wav_file = wave.open(output_filename, 'w')
wav_file.setnchannels(n_channels)
wav_file.setsampwidth(sampwidth)
wav_file.setframerate(sample_rate)

# --- Chunk Processing Setup ---
chunk_duration = 1.0  # Process audio 1 second at a time
chunk_samples = int(chunk_duration * sample_rate)  # Number of samples per chunk
total_chunks = int(duration / chunk_duration)

# Initialize phases to maintain continuity between chunks
phase_left = 0.0
phase_right = 0.0

# --- Main Processing Loop ---
for chunk_index in range(total_chunks):
    # Global time for this chunk (from t_start to t_start + chunk_duration)
    t_start = chunk_index * chunk_duration
    # Create a time array for the current chunk
    t_chunk = np.linspace(t_start, t_start + chunk_duration, chunk_samples, endpoint=False)

    # Get instantaneous frequencies for left and right channels
    f_left = left_frequency(t_chunk)
    f_right = right_frequency(t_chunk)

    # Calculate phase increments for each sample: d_phase = 2*pi*f/sample_rate
    dphase_left = 2 * np.pi * f_left / sample_rate
    dphase_right = 2 * np.pi * f_right / sample_rate

    # Accumulate phase increments to ensure continuous sine waves between chunks
    phase_array_left = phase_left + np.cumsum(dphase_left)
    phase_array_right = phase_right + np.cumsum(dphase_right)

    # Update the starting phase for the next chunk
    phase_left = phase_array_left[-1]
    phase_right = phase_array_right[-1]

    # Generate the sine waves for this chunk
    chunk_left = np.sin(phase_array_left)
    chunk_right = np.sin(phase_array_right)

    # Apply the fade envelope for smooth fade-in/out
    env = envelope(t_chunk, duration)
    chunk_left *= env
    chunk_right *= env

    # Scale the samples and convert to 16-bit integers
    chunk_left = (chunk_left * max_amplitude).astype(np.int16)
    chunk_right = (chunk_right * max_amplitude).astype(np.int16)

    # Interleave the two channels (stereo)
    interleaved = np.empty(chunk_samples * 2, dtype=np.int16)
    interleaved[0::2] = chunk_left
    interleaved[1::2] = chunk_right

    # Write the current chunk's frames to the WAV file
    wav_file.writeframes(interleaved.tobytes())

import numpy as np
import wave

# --- Constants and Parameters ---
sample_rate = 44100  # Samples per second
duration = 3600  # Total duration in seconds (60 minutes)
n_channels = 2  # Stereo
sampwidth = 2  # 16-bit PCM (2 bytes per sample)
output_filename = 'binaural_entrainment_hour.wav'

# Frequency design parameters
base_freq = 108  # Starting frequency in Hz for both ears
freq_shift = 30.0  # Amount of frequency shift (±20 Hz)
midpoint = duration / 2.0  # 1800 seconds; at midpoint, left is at 56 Hz and right at 16 Hz

# Fade parameters (in seconds)
fade_duration = 10.0  # 10-second fade-in and fade-out

# Maximum amplitude (scaled to 90% of full scale to prevent clipping)
max_amplitude = 32767 * 0.9

# --- Frequency Functions ---
def left_frequency(t):
    """ Returns the instantaneous left-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency increases linearly from base_freq to (base_freq+freq_shift).
    For midpoint < t <= duration, the frequency decreases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq + (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq + freq_shift) - (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


def right_frequency(t):
    """ Returns the instantaneous right-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency decreases linearly from base_freq to (base_freq-freq_shift).
    For midpoint < t <= duration, the frequency increases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq - (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq - freq_shift) + (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


# --- Envelope Function ---
def envelope(t, total_duration):
    """ Returns an amplitude envelope for time array t with a linear fade-in during the
    first fade_duration seconds and a fade-out during the final fade_duration seconds.
    """
    env = np.ones_like(t)  # Fade-in: scale linearly from 0 to 1 over fade_duration seconds
    mask_in = t < fade_duration
    env[mask_in] = t[mask_in] / fade_duration
    # Fade-out: scale linearly from 1 to 0 over the last fade_duration seconds
    mask_out = t > (total_duration - fade_duration)
    env[mask_out] = (total_duration - t[mask_out]) / fade_duration
    return env


# --- Prepare WAV File for Writing ---
wav_file = wave.open(output_filename, 'w')
wav_file.setnchannels(n_channels)
wav_file.setsampwidth(sampwidth)
wav_file.setframerate(sample_rate)

# --- Chunk Processing Setup ---
chunk_duration = 1.0  # Process audio 1 second at a time
chunk_samples = int(chunk_duration * sample_rate)  # Number of samples per chunk
total_chunks = int(duration / chunk_duration)

# Initialize phases to maintain continuity between chunks
phase_left = 0.0
phase_right = 0.0

# --- Main Processing Loop ---
for chunk_index in range(total_chunks):
    # Global time for this chunk (from t_start to t_start + chunk_duration)
    t_start = chunk_index * chunk_duration
    # Create a time array for the current chunk
    t_chunk = np.linspace(t_start, t_start + chunk_duration, chunk_samples, endpoint=False)

    # Get instantaneous frequencies for left and right channels
    f_left = left_frequency(t_chunk)
    f_right = right_frequency(t_chunk)

    # Calculate phase increments for each sample: d_phase = 2*pi*f/sample_rate
    dphase_left = 2 * np.pi * f_left / sample_rate
    dphase_right = 2 * np.pi * f_right / sample_rate

    # Accumulate phase increments to ensure continuous sine waves between chunks
    phase_array_left = phase_left + np.cumsum(dphase_left)
    phase_array_right = phase_right + np.cumsum(dphase_right)

    # Update the starting phase for the next chunk
    phase_left = phase_array_left[-1]
    phase_right = phase_array_right[-1]

    # Generate the sine waves for this chunk
    chunk_left = np.sin(phase_array_left)
    chunk_right = np.sin(phase_array_right)

    # Apply the fade envelope for smooth fade-in/out
    env = envelope(t_chunk, duration)
    chunk_left *= env
    chunk_right *= env

    # Scale the samples and convert to 16-bit integers
    chunk_left = (chunk_left * max_amplitude).astype(np.int16)
    chunk_right = (chunk_right * max_amplitude).astype(np.int16)

    # Interleave the two channels (stereo)
    interleaved = np.empty(chunk_samples * 2, dtype=np.int16)
    interleaved[0::2] = chunk_left
    interleaved[1::2] = chunk_right

    # Write the current chunk's frames to the WAV file
    wav_file.writeframes(interleaved.tobytes())

wav_file.close()
print("WAV file generated:", output_filename)
# Get instantaneous frequencies for left and right channels
f_left = left_frequency(t_chunk)
f_right = right_frequency(t_chunk)

# Calculate phase increments for each sample: d_phase = 2*pi*f/sample_rate
dphase_left = 2 * np.pi * f_left / sample_rate
dphase_right = 2 * np.pi * f_right / sample_rate

# Accumulate phase increments to ensure continuous sine waves between chunks
phase_array_left = phase_left + np.cumsum(dphase_left)
phase_array_right = phase_right + np.cumsum(dphase_right)

# Update the starting phase for the next chunk
phase_left = phase_array_left[-1]
phase_right = phase_array_right[-1]

# Generate the sine waves for this chunk
chunk_left = np.sin(phase_array_left)
chunk_right = np.sin(phase_array_right)

# Apply the fade envelope for smooth fade-in/out
env = envelope(t_chunk, duration)
chunk_left *= env
chunk_right *= env

# Scale the samples and convert to 16-bit integers
chunk_left = (chunk_left * max_amplitude).astype(np.int16)
chunk_right = (chunk_right * max_amplitude).astype(np.int16)

# Interleave the two channels (stereo)
interleaved = np.empty(chunk_samples * 2, dtype=np.int16)
interleaved[0::2] = chunk_left
interleaved[1::2] = chunk_right

import numpy as np
import wave

# --- Constants and Parameters ---
sample_rate = 44100  # Samples per second
duration = 3600  # Total duration in seconds (60 minutes)
n_channels = 2  # Stereo
sampwidth = 2  # 16-bit PCM (2 bytes per sample)
output_filename = 'binaural_entrainment_hour.wav'

# Frequency design parameters
base_freq = 108  # Starting frequency in Hz for both ears
freq_shift = 30.0  # Amount of frequency shift (±20 Hz)
midpoint = duration / 2.0  # 1800 seconds; at midpoint, left is at 56 Hz and right at 16 Hz

# Fade parameters (in seconds)
fade_duration = 10.0  # 10-second fade-in and fade-out

# Maximum amplitude (scaled to 90% of full scale to prevent clipping)
max_amplitude = 32767 * 0.9

# --- Frequency Functions ---
def left_frequency(t):
    """ Returns the instantaneous left-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency increases linearly from base_freq to (base_freq+freq_shift).
    For midpoint < t <= duration, the frequency decreases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq + (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq + freq_shift) - (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


def right_frequency(t):
    """ Returns the instantaneous right-channel frequency for time t (in seconds).
    For 0 <= t <= midpoint, the frequency decreases linearly from base_freq to (base_freq-freq_shift).
    For midpoint < t <= duration, the frequency increases linearly back to base_freq.
    """
    t = np.array(t)
    f = np.empty_like(t)
    mask1 = t <= midpoint
    f[mask1] = base_freq - (freq_shift / midpoint) * t[mask1]
    mask2 = ~mask1
    f[mask2] = (base_freq - freq_shift) + (freq_shift / midpoint) * (t[mask2] - midpoint)
    return f


# --- Envelope Function ---
def envelope(t, total_duration):
    """ Returns an amplitude envelope for time array t with a linear fade-in during the
    first fade_duration seconds and a fade-out during the final fade_duration seconds.
    """
    env = np.ones_like(t)  # Fade-in: scale linearly from 0 to 1 over fade_duration seconds
    mask_in = t < fade_duration
    env[mask_in] = t[mask_in] / fade_duration
    # Fade-out: scale linearly from 1 to 0 over the last fade_duration seconds
    mask_out = t > (total_duration - fade_duration)
    env[mask_out] = (total_duration - t[mask_out]) / fade_duration
    return env


# --- Prepare WAV File for Writing ---
wav_file = wave.open(output_filename, 'w')
wav_file.setnchannels(n_channels)
wav_file.setsampwidth(sampwidth)
wav_file.setframerate(sample_rate)

# --- Chunk Processing Setup ---
chunk_duration = 1.0  # Process audio 1 second at a time
chunk_samples = int(chunk_duration * sample_rate)  # Number of samples per chunk
total_chunks = int(duration / chunk_duration)

# Initialize phases to maintain continuity between chunks
phase_left = 0.0
phase_right = 0.0

# --- Main Processing Loop ---
for chunk_index in range(total_chunks):
    # Global time for this chunk (from t_start to t_start + chunk_duration)
    t_start = chunk_index * chunk_duration
    # Create a time array for the current chunk
    t_chunk = np.linspace(t_start, t_start + chunk_duration, chunk_samples, endpoint=False)

    # Get instantaneous frequencies for left and right channels
    f_left = left_frequency(t_chunk)
    f_right = right_frequency(t_chunk)

    # Calculate phase increments for each sample: d_phase = 2*pi*f/sample_rate
    dphase_left = 2 * np.pi * f_left / sample_rate
    dphase_right = 2 * np.pi * f_right / sample_rate

    # Accumulate phase increments to ensure continuous sine waves between chunks
    phase_array_left = phase_left + np.cumsum(dphase_left)
    phase_array_right = phase_right + np.cumsum(dphase_right)

    # Update the starting phase for the next chunk
    phase_left = phase_array_left[-1]
    phase_right = phase_array_right[-1]

    # Generate the sine waves for this chunk
    chunk_left = np.sin(phase_array_left)
    chunk_right = np.sin(phase_array_right)

    # Apply the fade envelope for smooth fade-in/out
    env = envelope(t_chunk, duration)
    chunk_left *= env
    chunk_right *= env

    # Scale the samples and convert to 16-bit integers
    chunk_left = (chunk_left * max_amplitude).astype(np.int16)
    chunk_right = (chunk_right * max_amplitude).astype(np.int16)

    # Interleave the two channels (stereo)
    interleaved = np.empty(chunk_samples * 2, dtype=np.int16)
    interleaved[0::2] = chunk_left
    interleaved[1::2] = chunk_right

    # Write the current chunk's frames to the WAV file
    wav_file.writeframes(interleaved.tobytes())

wav_file.close()
print("WAV file generated:", output_filename)

WAV file generated: binaural_entrainment_hour.wav
WAV file generated: binaural_entrainment_hour.wav
WAV file generated: binaural_entrainment_hour.wav
WAV file generated: binaural_entrainment_hour.wav
